<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="../images/nvidia.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building LLM Applications With Prompt Engineering</b></font></h1>
<h2><b>Chain-of-Thought Prompting</b></h2>
<br>


## License

Unless otherwise noted:

- Code cells are licensed under the
  [Apache License 2.0](../../../LICENSES/Apache-2.0.txt).
- Markdown and raw cells are licensed under the
  [Creative Commons Attribution 4.0 International License](../../../LICENSES/CC-BY-4.0.txt).
- Third-party material retains its original license and is identified in
  [`THIRD_PARTY_NOTICES.md`](../../../THIRD_PARTY_NOTICES.md).

Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES.

In this notebook, you will learn about one of the most famous prompting techniques called Chain-of-Thought prompting.

---

## Objectives

By the time you complete this notebook you will:

- Learn Chain-of-Thought prompting
- Encounter and appreciate LLM hallucination

---

## Imports

In [ ]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

---

## Create a Model Instance

### API key for this notebook kernel

Paste the API key for your configured endpoint when the next cell prompts for it. The typed value is visible during entry but is not placed in the cell source or written to a file. It remains only in this kernel's memory. Restarting the kernel discards it, and a separate notebook kernel will ask for the key again.

Avoid entering the key while screen sharing or recording.

In [ ]:
import os

if not os.environ.get("NVIDIA_API_KEY"):
    _api_key = input("Paste the API key for the configured endpoint: ").strip()
    if not _api_key:
        raise ValueError("An API key is required to run this notebook.")
    os.environ["NVIDIA_API_KEY"] = _api_key
    del _api_key

os.environ.setdefault("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")

# To use an alternative endpoint, uncomment below and change the URL.
# os.environ["NVIDIA_BASE_URL"] = "https://your-endpoint.example.com/v1"
print("API access is configured for this notebook kernel.")

In [ ]:
base_url = os.getenv("NVIDIA_BASE_URL")
model = 'nvidia/nemotron-3.5-lightning-30b-a3b'
# For an alternative endpoint, use the model ID accepted by that endpoint.
# model = 'your-model-id'
llm = ChatNVIDIA(
    base_url=base_url, model=model, temperature=0,
)

---

## LLMs Jump to Conclusions

Since LLMs are primarily designed to generate what is most likely to come next in a stream of text, it's no surprise that they can often "jump to conclusions" in scenarios that actually require complex reasoning.

Consider the following example, which challenges the LLM to do some multiplication. Before invoking the LLM, we'll use Python to provide us with the actual answer.

In [ ]:
expression1 = "345 * 88888"
expression2 = "345*88888"
eval(expression1), eval(expression2)

In [ ]:
print(llm.invoke(f'What is {expression1}? Give me just the answer').content)
print(llm.invoke(f'What is {expression2}? Give me just the answer').content)

Compare both model answers with the Python result above. Any difference is visible evidence that fluent output is not arithmetic verification.

Keep independent, checkable evidence for tasks requiring exact calculations. The next cells compare whether Chain-of-Thought prompts change the approach, while Python's result remains the correctness check.

---

## Hallucination

This phenomenon of LLMs generating incorrect information, often with what appears like "confidence" to outside observers, is called **hallucination**. Because of their design, an LLM will usually say something incorrect (unbeknownst to the LLM itself) rather than say nothing at all. While it's true that LLMs over time are hallucinating less, and that over time they will continue to improve in this regard, hallucination is an inherent characteristic of LLM generation.

Regarding LLM hallucination, you need to commit the following golden rules to memory:
1. All LLMs hallucinate.
2. You are responsible for what your LLMs generate in your applications.

---

## Chain-of-Thought Prompting

A prompt can ask a model to show intermediate steps, giving us more of its answer to inspect. Those steps still need verification.

[**Chain-of-Thought Prompting Elicits Reasoning in Large Language Models**](https://arxiv.org/abs/2201.11903) describes using worked examples that solve problems step by step. The following course-created example illustrates the prompting pattern.

### Course-created example

Suppose a workshop has 8 tables with 6 seats each, and 5 seats are empty.

| Prompting style | Example prompt |
|---|---|
| Direct | `How many learners are seated? Return only the answer.` |
| Step-by-step | `How many learners are seated? Show the intermediate calculation, then give the final answer.` |

The second prompt encourages the model to make the useful intermediate calculation explicit: there are $8 \times 6 = 48$ seats in total, and $48 - 5 = 43$ learners are seated.

As you can see in the right-hand-side example, by modeling for the LLM that it ought to break the complex task into smaller subtasks that it is more likely to be able to perform, the model is given "time to think" and can do better at tasks requiring complex reasoning.

---

## Chain-of-Though Multiplication

Let's try a CoT prompt for 3-digit long multiplication. We will start with an example multiplication prompt.

In [ ]:
example_problem = 'What is 678 * 789?'

Next, we'll supply a chain-of-thought example of working through the problem step by step.

In [ ]:
example_cot = '''\
Let me break this down into steps. First I'll break down 789 into hundreds, tens, and ones:

789 -> 700 + 80 + 9

Next I'll multiply 678 by each of these values, storing the intermediate results:

678 * 700 -> 678 * 7 * 100 -> 4746 * 100 -> 474600

My first intermediate result is 474600.

678 * 80 -> 678 * 8 * 10 -> 5424 * 10 -> 54240

My second intermediate result is 54240.

678 * 9 -> 6102

My third intermediate result is 6102.

My three intermediate results are 474600, 54240, and 6102.

Adding the first two intermediate results I get 474600 + 54240 -> 528840.

Adding 528840 to the last intermediate result I get 528840 + 6102 -> 534942

The final result is 534942.
'''

With our example problem and example CoT response, we can construct a one-shot prompt template.

In [ ]:
multiplication_template = ChatPromptTemplate.from_messages([
    ('human', example_problem),
    ('ai', example_cot),
    ('human', '{long_multiplication_prompt}')
])

We'll now use this template in a simple chain.

In [ ]:
multiplication_chain = multiplication_template | llm | StrOutputParser()

Let's revisit the original multiplication problem we tasked the LLM with to see if it does any better.

In [ ]:
print(multiplication_chain.invoke(f'What is {expression1}?'))
# print(multiplication_chain.invoke(f'What is {expression2}?'))
# print(multiplication_chain.invoke(f'What is {expression2+"88888"}?'))

In [ ]:
print(f"{expression1} = {eval(expression1)}")
print(f"{expression2} = {eval(expression2)}")
print(f"{expression2+'88888'} = {eval(expression2+'88888'):,}")

Check whether the response follows the example's step-by-step form. Compare its final value with Python's result above: a reasoning-shaped response does not establish that the arithmetic is correct.

---

## Zero-shot Chain-of-Thought Prompting

CoT prompting can get really verbose, and as we mentioned earlier, you shouldn't be shy about writing long prompts when needed to accomplish your goals.

That said, there are more elegant ways to leverage CoT methods with LLMs. One such variant is called [Zero-shot CoT](https://arxiv.org/abs/2205.11916). This prompting technique simply adds "Let's think step by step" to the prompt without supplying verbose CoT examples, and it has been shown to work quite well.

Let's construct a new prompt template to try Zero-shot CoT on long multiplication, following the example in the paper, simply supplying the language "Let's think step by step".

In [ ]:
zero_shot_cot_prompt = ChatPromptTemplate([
    ("human", "{long_multiplication_prompt} Let's think step by step.")
])

In [ ]:
zero_shot_multiplication_chain = zero_shot_cot_prompt | llm | StrOutputParser()

In [ ]:
print(zero_shot_multiplication_chain.invoke(f'What is {expression1}?'))
# print(zero_shot_multiplication_chain.invoke(f'What is {expression2}?'))
# print(zero_shot_multiplication_chain.invoke(f'What is {expression2+"88888"}?'))

In [ ]:
print(f"{expression1} = {eval(expression1):,}")
print(f"{expression2} = {eval(expression2):,}")
print(f"{expression2+'88888'} = {eval(expression2+'88888'):,}")

Inspect the zero-shot response for intermediate steps and a final answer. Compare it with Python's result above: more reasoning text can still contain flawed steps or exhaust the response budget before reaching an answer.

---

## CoT Prompting In Practice

As with many other aspects of prompt engineering, especially given the variety of LLMs available (both now and in the future) it's difficult to give hard and fast rules about when and how exactly to employ CoT prompting. That said, we can offer you some good general guidelines to follow.

- Develop your prompts iteratively. Start simple, try zero-shot CoT when you think it might be helpful, and expand to use more verbose example-based CoT prompting when the need arises.
- Consider the use of external, non-LLM tools for tasks that LLMs may not be well-suited for (like math). We are going to look at tool use later in the workshop but the general idea is that LLMs are incredible, but not necessarily for every single task. Reflect if you will on the case of long multiplication about how fast, reliable, and effective simple Python is. Don't fall into the old adage _"If all you have is a hammer, everything looks like a nail."_ LLMs are only one of your many tools to use while building LLM-based applications.

---

## Exercise: Use an LLM to Solve a Word Problem

For this exercise, use what you've learned in this notebook to a correct response back from our LLM to the following word problem.

In [ ]:
word_problem = (
    "Michael's car travels at 40 miles per hour. He is driving from 1 PM to 4 PM"
    "and then travels back at a rate of 25 miles per hour due to heavy traffic."
    " How long in terms of minutes did it take him to get back?"
)

Before beginning your work, here is the actual solution to the word problem.

<blockquote>
Michael drove 40 miles per hour for 3 hours, which means he drove 120 miles.

To come back at a rate of 25 miles per hour, it would have taken him 4.8 hours (120 miles / 25 mph) which is equivalent to **288** minutes (4.8 hours * 60 minutes/hour).

The correct answer is **288**.
</blockquote>

Feel free to check out the *Solution* below if you get stuck.

### Your Work Here

### Solution

There are lots of ways to solve this, but we opted to include a system message encouraging the LLM to always break their work into smaller tasks and showing their work, and then, appending the cannonical zero-shot CoT prompt to whatever the human user passes in.

In [ ]:
template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert word problem solver. You always break your problem down into smaller tasks and show your work.'),
    ('human', '{prompt}\n\nLet\'s think step by step.')
])

In [ ]:
chain = template | llm | StrOutputParser()

In [ ]:
print(chain.invoke(word_problem))

---

## Summary

You compared direct answers with Chain-of-Thought prompts and checked the results independently. Record which prompts changed the approach and whether those changes improved correctness.

The next notebook compares thinking enabled and disabled on the same Lightning model.

---

<!-- NEXT_STEP_CARD -->

<div style="border-left: 6px solid #76B900; background: #f7fdf2; padding: 14px 18px; border-radius: 10px; margin: 20px 0;">
<p style="margin: 0 0 6px; color: #315c00; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;">Continue the unified course path</p>
<p style="margin: 0 0 8px; color: black;"><strong>Next step:</strong> Open <code>3-Prompting-With-Messages/35-Reasoning-Models.ipynb</code> next: <strong>Reasoning Models</strong>.</p>
<p style="margin: 0; color: black;"><strong>Before moving on:</strong> Save one task and its verified answer so you can compare thinking modes on the same input in the next notebook.</p>
</div>